# Cox Proportional Hazards

Hazard model fit on the same feature set as the classifiers (age, sex, 12 comorbidity categories, 5 utilisation measures). Coefficients are reported as hazard ratios (HR = exp(b)); the proportional-hazards assumption is tested via Schoenfeld residuals.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import CoxPHFitter, KaplanMeierFitter
import warnings
warnings.filterwarnings('ignore')

BASE = '../../'  
USYD = ['#E64626', '#DAA8A2', '#71A399', '#8F9EC9', '#1A345E']  # same palette as the Markov chapter's figures

df = pd.read_csv(BASE + 'modeling_dataset.csv')
print(f'{len(df):,} patients, {df.incident_cvd.sum():,} incident CVD events ({100*df.incident_cvd.mean():.1f}%)')
print(f'years_followup: min={df.years_followup.min():.2f}, median={df.years_followup.median():.2f}, max={df.years_followup.max():.2f}')

3,050 patients, 472 incident CVD events (15.5%)
years_followup: min=0.00, median=2.89, max=18.54


## 1. Covariates

Same feature set as the classifiers: age, sex, 12 comorbidity categories, 5 baseline utilisation measures.

In [2]:
COMORBID_COLS = ['cat_Renal', 'cat_FluidElectrolyte', 'cat_Metabolic', 'cat_Respiratory',
                  'cat_MentalHealthSubstance', 'cat_Musculoskeletal', 'cat_Hematologic', 'cat_Hepatic',
                  'cat_Neurologic', 'cat_Oncologic', 'cat_EndocrineOther', 'cat_ConstitutionalFrailty']
UTILIZATION_COLS = ['baseline_n_diagnoses', 'baseline_n_procedures', 'baseline_n_claims',
                     'baseline_n_episodes', 'baseline_history_days']

cox_df = df[['years_followup', 'incident_cvd', 'age'] + COMORBID_COLS + UTILIZATION_COLS].copy()
cox_df['sex_M'] = (df['sex'] == 'M').astype(int)
cox_df['incident_cvd'] = cox_df['incident_cvd'].astype(int)

COVARIATES = ['age', 'sex_M'] + COMORBID_COLS + UTILIZATION_COLS
print(f'{len(COVARIATES)} covariates: {COVARIATES}')
print(f'{len(cox_df):,} patients, {cox_df.incident_cvd.sum():,} events')
print(f'duration==0 rows: {(cox_df.years_followup == 0).sum()} (all should be immediate events, not censoring)')
assert ((cox_df['years_followup'] == 0) & (cox_df['incident_cvd'] == 0)).sum() == 0, 'zero-duration censored row would be undefined for Cox'
cox_df.head()

19 covariates: ['age', 'sex_M', 'cat_Renal', 'cat_FluidElectrolyte', 'cat_Metabolic', 'cat_Respiratory', 'cat_MentalHealthSubstance', 'cat_Musculoskeletal', 'cat_Hematologic', 'cat_Hepatic', 'cat_Neurologic', 'cat_Oncologic', 'cat_EndocrineOther', 'cat_ConstitutionalFrailty', 'baseline_n_diagnoses', 'baseline_n_procedures', 'baseline_n_claims', 'baseline_n_episodes', 'baseline_history_days']
3,050 patients, 472 events
duration==0 rows: 131 (all should be immediate events, not censoring)


,years_followup,incident_cvd,age,cat_Renal,cat_FluidElectrolyte,cat_Metabolic,cat_Respiratory,cat_MentalHealthSubstance,cat_Musculoskeletal,cat_Hematologic,...,cat_Neurologic,cat_Oncologic,cat_EndocrineOther,cat_ConstitutionalFrailty,baseline_n_diagnoses,baseline_n_procedures,baseline_n_claims,baseline_n_episodes,baseline_history_days,sex_M
0,0.00000,1,95,0,0,0,0,0,0,0,...,0,0,0,0,18,0,2,2,4430,0
1,16.15332,0,106,0,0,0,0,0,0,0,...,0,0,0,0,4,2,1,1,0,1
2,18.53525,0,102,0,0,0,1,0,0,0,...,0,1,0,0,12,4,3,3,0,0
3,0.00000,1,102,0,0,0,0,0,0,0,...,0,0,0,0,4,2,1,1,0,0
4,18.53525,0,101,0,0,0,0,0,0,0,...,0,0,0,0,17,10,5,5,0,1


## 2. Fit the Cox model

In [3]:
cph = CoxPHFitter()
cph.fit(cox_df, duration_col='years_followup', event_col='incident_cvd', formula=' + '.join(COVARIATES))
print(f'Concordance index (Cox, in-sample): {cph.concordance_index_:.4f}')
print(f'Log-likelihood ratio test: {cph.log_likelihood_ratio_test()}')

Concordance index (Cox, in-sample): 0.7005
Log-likelihood ratio test: <lifelines.StatisticalResult: log-likelihood ratio test>
null_distribution = chi squared
  degrees_freedom = 19
        test_name = log-likelihood ratio test

---
 test_statistic      p  -log2(p)
         196.24 <0.005    102.05


## 3. Hazard ratios with 95% CIs

In [4]:
hr_table = cph.summary[['coef', 'exp(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'p']].copy()
hr_table.columns = ['coef', 'hazard_ratio', 'hr_ci_lower', 'hr_ci_upper', 'p_value']
hr_table = hr_table.sort_values('hazard_ratio', ascending=False)
hr_table.to_csv('cox_hazard_ratios.csv')
print('=== Hazard ratios, sorted, with 95% CI ===')
hr_table.round(4)

=== Hazard ratios, sorted, with 95% CI ===


,coef,hazard_ratio,hr_ci_lower,hr_ci_upper,p_value
covariate,,,,,
cat_Metabolic,1.1790,3.2512,2.1594,4.8950,0.0000
cat_Renal,0.9747,2.6504,1.5661,4.4854,0.0003
cat_FluidElectrolyte,0.8237,2.2790,1.4532,3.5741,0.0003
cat_Hematologic,0.5865,1.7976,0.8867,3.6444,0.1039
cat_MentalHealthSubstance,0.5400,1.7160,0.8694,3.3872,0.1196
cat_Neurologic,0.4496,1.5677,0.8888,2.7652,0.1204
sex_M,0.0934,1.0979,0.9110,1.3231,0.3265
age,0.0355,1.0362,1.0288,1.0436,0.0000
cat_Musculoskeletal,0.0339,1.0345,0.7689,1.3918,0.8229


## 4. Testing the proportional-hazards assumption (Schoenfeld residuals)

In [ ]:
import io, contextlib

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    try:
        cph.check_assumptions(cox_df, p_value_threshold=0.05, show_plots=False)
    except Exception as e:
        print(f'check_assumptions raised: {e}')
assumption_output = buf.getvalue()
print(assumption_output)

with open('cox_schoenfeld_assumption_check.txt', 'w') as f:
    f.write(assumption_output)

In [6]:
from lifelines.statistics import proportional_hazard_test

ph_test = proportional_hazard_test(cph, cox_df, time_transform='rank')
ph_test_df = ph_test.summary.reset_index().rename(columns={'index': 'covariate'})
ph_test_df.to_csv('cox_schoenfeld_test.csv', index=False)
print('=== Schoenfeld residual test per covariate (rank time transform) ===')
print('A significant p-value (p<0.05) means that covariate\'s effect is NOT constant over time -- a PH violation for that specific variable.')
ph_test_df.round(4)

=== Schoenfeld residual test per covariate (rank time transform) ===
A significant p-value (p<0.05) means that covariate's effect is NOT constant over time -- a PH violation for that specific variable.


,covariate,test_statistic,p,-log2(p)
0,age,24.7858,0.0000,20.5739
1,baseline_history_days,4.1573,0.0415,4.5923
2,baseline_n_claims,0.5630,0.4531,1.1422
3,baseline_n_diagnoses,4.1442,0.0418,4.5811
4,baseline_n_episodes,2.6428,0.1040,3.2650
5,baseline_n_procedures,0.0004,0.9843,0.0229
6,cat_ConstitutionalFrailty,0.0048,0.9449,0.0818
7,cat_EndocrineOther,0.0000,0.9996,0.0005
8,cat_FluidElectrolyte,4.2637,0.0389,4.6828
9,cat_Hematologic,0.0007,0.9785,0.0314
